# Library Imports

In [1]:
import pandas as pd

# Importing Evaluation Dataframes

In [3]:
# Load the parquet file into a DataFrame
baseline_no_pred_eval = pd.read_parquet('/content/baseline_no_pred_eval.parquet')
baseline_w_pred_eval = pd.read_parquet('/content/baseline_w_pred_eval.parquet')
eval_autonbeats = pd.read_parquet('/content/eval_autonbeats_1.parquet')
eval_autonhits = pd.read_parquet('/content/eval_autonhits_1.parquet')
eval_lgb = pd.read_parquet('/content/eval_lgb.parquet')
eval_tcf = pd.read_parquet('/content/eval_tcf_1.parquet')

In [ ]:
display(baseline_w_pred_eval.head(n=6))

display(eval_autonbeats.head(n=6))

display(eval_autonhits.head(n=6))

display(eval_lgb.head(n=6))

display(eval_tcf.head(n=6))

,unique_id,metric,AutoARIMAwPred
0,hotel_0,bias,0.034148
1,hotel_0,mae,0.127840
2,hotel_0,mape,0.205180
3,hotel_0,rmse,0.158891
4,hotel_105,bias,-0.006306
5,hotel_105,mae,0.071678


,unique_id,metric,AutoNBEATS
0,hotel_0,bias,-0.004609
1,hotel_0,mae,0.161878
2,hotel_0,mape,0.261339
3,hotel_0,rmse,0.195913
4,hotel_105,bias,-0.090817
5,hotel_105,mae,0.113671


,unique_id,metric,AutoNHITS
0,hotel_0,bias,0.000079
1,hotel_0,mae,0.164236
2,hotel_0,mape,0.267327
3,hotel_0,rmse,0.198349
4,hotel_105,bias,-0.085752
5,hotel_105,mae,0.115697


,unique_id,metric,LGBMRegressor
0,hotel_0,bias,0.043267
1,hotel_0,mae,0.110940
2,hotel_0,mape,0.186018
3,hotel_0,rmse,0.136007
4,hotel_105,bias,-0.036938
5,hotel_105,mae,0.079631


,unique_id,metric,Chronos,Moirai,TimesFM-2.5
0,hotel_0,bias,0.057866,0.112502,0.074234
1,hotel_0,mae,0.155222,0.169799,0.133578
2,hotel_0,mape,0.274553,0.312501,0.241742
3,hotel_0,rmse,0.187122,0.212194,0.166941
4,hotel_105,bias,-0.025065,-0.009047,-0.022219
5,hotel_105,mae,0.087805,0.095786,0.088886


# Combining Evaluation DataFrames

In [4]:
combined_eval_df = baseline_no_pred_eval.copy()

# Merge baseline_w_pred_eval
combined_eval_df = pd.merge(combined_eval_df, baseline_w_pred_eval, on=['unique_id', 'metric'], how='left')

# Merge eval_autonbeats
combined_eval_df = pd.merge(combined_eval_df, eval_autonbeats, on=['unique_id', 'metric'], how='left')

# Merge eval_autonhits
combined_eval_df = pd.merge(combined_eval_df, eval_autonhits, on=['unique_id', 'metric'], how='left')

# Merge eval_lgb
combined_eval_df = pd.merge(combined_eval_df, eval_lgb, on=['unique_id', 'metric'], how='left')

# Merge eval_tcf
combined_eval_df = pd.merge(combined_eval_df, eval_tcf, on=['unique_id', 'metric'], how='left')

display(combined_eval_df.head())

,unique_id,metric,Naive,SeasonalNaive,AutoETS,AutoARIMA,AutoARIMAwPred,AutoNBEATS,AutoNHITS,LGBMRegressor,Chronos,Moirai,TimesFM-2.5
0,hotel_0,bias,0.082011,0.029806,-0.007384,0.047858,0.034148,-0.004609,0.000079,0.043267,0.057866,0.112502,0.074234
1,hotel_0,mae,0.194709,0.214286,0.184314,0.167087,0.127840,0.161878,0.164236,0.110940,0.155222,0.169799,0.133578
2,hotel_0,mape,0.341090,0.354191,0.297296,0.286336,0.205180,0.261339,0.267327,0.186018,0.274553,0.312501,0.241742
3,hotel_0,rmse,0.238226,0.252785,0.216098,0.198841,0.158891,0.195913,0.198349,0.136007,0.187122,0.212194,0.166941
4,hotel_105,bias,0.053190,-0.073854,-0.062427,-0.059772,-0.006306,-0.090817,-0.085752,-0.036938,-0.025065,-0.009047,-0.022219


In [5]:
# Save as CSV
combined_eval_df.to_csv("combined_eval_df.csv", index=False)

# Determining the Best Model

I will use MAE as my metric to evaluate the models on since it measures the average magnitude of errors, treating overpredicting and underpredicting with equal penalization by ignoring the direction and taking the absolute value, additionally MAE is highly interpretable, as the resulting error is expressed directly in the same unit as the target (actual room demand).

In [6]:
selected_metric = 'mae'

# Filter to that metric only
metric_eval_df = combined_eval_df[combined_eval_df['metric'] == selected_metric].copy()

# Drop metrics column
metric_eval_df = metric_eval_df.drop(columns=['metric'])

# Define model names, exclude unique_id from this
model_columns = metric_eval_df.columns.drop('unique_id').tolist()

# Find the best model for each id seeking a small MAE
metric_eval_df['best_model'] = metric_eval_df[model_columns].idxmin(axis=1)

# Pull the winner model for each hotel
evaluation_df_for_selection = metric_eval_df[['unique_id', 'best_model']]

evaluation_df_for_selection.head(17)

,unique_id,best_model
1,hotel_0,LGBMRegressor
5,hotel_105,AutoARIMAwPred
9,hotel_112,TimesFM-2.5
13,hotel_126,TimesFM-2.5
17,hotel_133,AutoARIMAwPred
21,hotel_14,AutoARIMAwPred
25,hotel_21,TimesFM-2.5
29,hotel_35,LGBMRegressor
33,hotel_42,LGBMRegressor
37,hotel_49,AutoARIMAwPred


In [7]:
# Save as Parquet
evaluation_df_for_selection.to_parquet('bestmodel_perhotel.parquet', index=False)

In [11]:
# Calculate how many times each model is the best for each unique_id
best_model_counts = evaluation_df_for_selection.groupby('unique_id')['best_model'].value_counts().unstack(fill_value=0)


all_models = model_columns # Assuming model_columns is defined and contains all model names
best_model_counts = best_model_counts.reindex(columns=all_models, fill_value=0)

display(best_model_counts.head(100))

best_model,Naive,SeasonalNaive,AutoETS,AutoARIMA,AutoARIMAwPred,AutoNBEATS,AutoNHITS,LGBMRegressor,Chronos,Moirai,TimesFM-2.5
unique_id,,,,,,,,,,,
hotel_0,0,0,0,0,0,0,0,1,0,0,0
hotel_105,0,0,0,0,1,0,0,0,0,0,0
hotel_112,0,0,0,0,0,0,0,0,0,0,1
hotel_126,0,0,0,0,0,0,0,0,0,0,1
hotel_133,0,0,0,0,1,0,0,0,0,0,0
hotel_14,0,0,0,0,1,0,0,0,0,0,0
hotel_21,0,0,0,0,0,0,0,0,0,0,1
hotel_35,0,0,0,0,0,0,0,1,0,0,0
hotel_42,0,0,0,0,0,0,0,1,0,0,0


In [12]:
# Sum all the wins across models
total_global_wins = best_model_counts.sum(axis=0)
print(total_global_wins)

best_model
Naive             1
SeasonalNaive     0
AutoETS           0
AutoARIMA         0
AutoARIMAwPred    6
AutoNBEATS        0
AutoNHITS         0
LGBMRegressor     6
Chronos           0
Moirai            0
TimesFM-2.5       4
dtype: int64


In [10]:
# Save as Parquet
total_global_wins.to_frame(name='win_count').to_parquet('wincount_permodel.parquet', index=True)

Given that AutoARIMAwpred and LGBM are tied, I selected **LGBM** as the final model to generate the 28-day demand forecasts for each hotel.

This decision is based on its superior processing speed, scalability, and ability to handle complex features. While AutoARIMA can handle covariates, LGBM can seamlessly leverage both external covariates and complex lag variables, while also offering the advantage of automated feature selection to figure out which variables matter most.